# Label break-even: fine-tune the Manhattan CNN on a new layout family (Nanovate, Part 1)

**Question:** how many labelled images of a new layout (via, curvy) does the Manhattan-trained CNN need before it matches the
training-free verifier? This notebook fine-tunes `m6_cnn/cnn.pt` on k labelled **synthetic** patches of each family's *train*
split, k in {0, 12, 24, 48, 96, 192, 384, 768}, 3 seeds each (42 fine-tunes). The verifier and the test split are not touched;
evaluation runs locally afterwards.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Run the cells **in order**; if the runtime restarts, run step 0 again.

Expected time: data generation on the Colab CPU (a few minutes per family), then about 20-30 min of GPU fine-tuning.
GPU training is not bit-reproducible; seeds, configs and loss histories are saved with the weights.

**Return:** the `breakeven.zip` downloaded in step 3.

In [ ]:
!nvidia-smi | head -15

## 0. Setup (clone the `day0-value-case` branch + install; safe to re-run)

In [ ]:
import os
if not os.path.exists('/content/Nanovate'):
    !git clone -q https://github.com/DevAriwala2712/Nanovate.git /content/Nanovate
%cd /content/Nanovate
!git fetch -q origin day0-value-case && git checkout -q day0-value-case && git pull -q origin day0-value-case
!git rev-parse HEAD | tee /content/commit.txt
!pip install -q -e .
!ls m6_cnn/cnn.pt scripts/breakeven_train.py

## 1. Smoke test (tiny pool, 5 steps, about a minute)

In [ ]:
%cd /content/Nanovate
!python scripts/breakeven_train.py --families via --k 0,12 --seeds 0 --pool-layouts 2 --pool-dies 1 --steps 5 --out /content/be_smoke

## 2. Full run

Pool: 32 layouts x 4 dies x 6 kinds = 768 patches per family. Each fine-tune is 300 optimiser steps (Adam, lr 5e-4, batch 32).
The manifest is rewritten after every run, so a run cut short still returns usable results.

In [ ]:
%cd /content/Nanovate
!python scripts/breakeven_train.py --out /content/breakeven
!cp /content/commit.txt /content/breakeven/commit.txt

## 3. Download the results

In [ ]:
!cd /content && zip -qr breakeven.zip breakeven
from google.colab import files
files.download('/content/breakeven.zip')